# Clase 190 — Uplift modeling y DiD (difference-in-differences)

Dos técnicas causales de industria con datos observacionales/panel: **DiD** (evolución antes/después en tratado vs control, vía OLS con interacción) y **uplift modeling** (predecir a quién conviene tratar, CATE individual con T-learner + Qini).

Requiere: `numpy`, `pandas`, `statsmodels`, `scikit-learn`, `matplotlib`.

## 1. DiD con OLS

`Y = β₀ + β₁·tratado + β₂·post + β₃·(tratado×post) + ε`. El coeficiente `β₃` es el efecto causal bajo **parallel trends**.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

n = 2000
treated = rng.integers(0, 2, n)
post = rng.integers(0, 2, n)
true_effect = 2.5
Y = 10 + 1.5*treated + 3.0*post + true_effect*(treated*post) + rng.normal(0, 2, n)
df = pd.DataFrame({"Y": Y, "treated": treated, "post": post})
m = smf.ols("Y ~ treated * post", data=df).fit()
beta3 = m.params["treated:post"]
ci = m.conf_int().loc["treated:post"]
print(f"DiD β3 = {beta3:.3f}  IC95%=({ci[0]:.3f}, {ci[1]:.3f})  verdadero={true_effect}")
assert abs(beta3 - true_effect) < 0.5

## 2. Event study: chequear parallel trends

Con varios períodos, la diferencia tratado-control debe ser ≈ 0 **antes** del tratamiento (pre-tendencias planas) y saltar después.

In [ ]:
periods = np.arange(-5, 6)            # -5..-1 pre, 0..5 post
unit_treated = np.repeat([0, 1], 200)
rows = []
for u, tr in enumerate(unit_treated):
    ai = rng.normal(0, 1)             # efecto fijo de unidad
    for tp in periods:
        eff = true_effect if (tr == 1 and tp >= 0) else 0.0
        rows.append((u, tr, tp, 5 + 0.2*tp + ai + eff + rng.normal(0, 1)))
pan = pd.DataFrame(rows, columns=["unit", "treated", "t", "y"])
diff = pan[pan.treated == 1].groupby("t").y.mean() - pan[pan.treated == 0].groupby("t").y.mean()
pre = diff.loc[-5:-1].abs().mean()
print(f"|dif| media pre-tratamiento = {pre:.2f}  (≈0 => parallel trends plausible)")
assert pre < 0.5

plt.figure(figsize=(7, 4))
plt.plot(diff.index, diff.values, "o-")
plt.axvline(-0.5, color="k", ls="--", label="inicio tratamiento")
plt.axhline(0, color="gray", lw=0.8)
plt.xlabel("período relativo"); plt.ylabel("dif. tratado - control")
plt.title("Event study: pre≈0, salto en post"); plt.legend()
plt.tight_layout(); plt.show()

## 3. Uplift con T-learner

Dos modelos separados (`μ₁` para tratados, `μ₀` para control); el uplift es `μ₁(x) - μ₀(x)`. Simulamos un efecto que crece con la feature `x`.

In [ ]:
N = 8000
x = rng.uniform(0, 1, N)
t = rng.integers(0, 2, N)
p = (0.15 + 0.2*x) + (0.5*x) * t          # uplift verdadero = 0.5*x
y = (rng.uniform(0, 1, N) < p).astype(int)
data = pd.DataFrame({"x": x, "t": t, "y": y})
Xt = data[["x"]].values

m1 = RandomForestClassifier(n_estimators=200, min_samples_leaf=50, random_state=42).fit(Xt[t == 1], y[t == 1])
m0 = RandomForestClassifier(n_estimators=200, min_samples_leaf=50, random_state=42).fit(Xt[t == 0], y[t == 0])
grid = np.linspace(0, 1, 100).reshape(-1, 1)
uplift_pred = m1.predict_proba(grid)[:, 1] - m0.predict_proba(grid)[:, 1]
corr = np.corrcoef(grid.ravel(), uplift_pred)[0, 1]
print(f"correlación(uplift_pred, x) = {corr:.3f}  (>0: capta que el uplift crece con x)")
assert corr > 0.5

## 4. Qini curve

Ordenamos a los individuos por uplift predicho y acumulamos la ganancia incremental frente a tratar al azar.

In [ ]:
data["uplift_hat"] = m1.predict_proba(Xt)[:, 1] - m0.predict_proba(Xt)[:, 1]
o = data.sort_values("uplift_hat", ascending=False).reset_index(drop=True)
cum_t  = (o.t == 1).cumsum().values
cum_c  = (o.t == 0).cumsum().values
cum_yt = (o.y * (o.t == 1)).cumsum().values
cum_yc = (o.y * (o.t == 0)).cumsum().values
ratio = np.divide(cum_t, np.maximum(cum_c, 1))
qini = cum_yt - cum_yc * ratio
k = np.arange(1, len(o) + 1)

plt.figure(figsize=(7, 4))
plt.plot(k, qini, label="modelo uplift")
plt.plot([0, len(o)], [0, qini[-1]], "k--", label="aleatorio")
plt.xlabel("# individuos tratados (ordenados por uplift)")
plt.ylabel("ganancia incremental"); plt.legend(); plt.title("Qini curve")
plt.tight_layout(); plt.show()
print(f"ganancia final acumulada = {qini[-1]:.1f}")

## 5. DiD vs diferencia ingenua

La diferencia post ingenua mezcla el efecto con la diferencia basal entre grupos; el DiD la aísla.

In [ ]:
naive_post = (df[(df.treated == 1) & (df.post == 1)].Y.mean()
              - df[(df.treated == 0) & (df.post == 1)].Y.mean())
print(f"diferencia ingenua post = {naive_post:.3f}  (mezcla efecto + dif. basal)")
print(f"DiD β3                  = {beta3:.3f}  (aísla el efecto causal ≈ 2.5)")
assert naive_post > beta3

## Ejercicios

1. Violá parallel trends dándole al control una tendencia distinta en el pre y observá cómo el event study lo delata.
2. Entrená un X-learner simple (ponderá los pseudo-efectos por propensity) y compará su uplift con el T-learner.
3. Evaluá el modelo con `uplift@k` (ganancia al tratar el top 20 %) además de la Qini completa.

## Conclusiones

- DiD estima el efecto como el coeficiente de la interacción `tratado×post`; su validez depende de **parallel trends**.
- Verificá parallel trends con un event study (pre-tendencias planas) antes de creerle al `β₃`.
- El uplift es CATE individual: se evalúa con Qini / uplift@k, **no** con AUC de clasificación.
- T/S/X-learner y causal forest estiman heterogeneidad del efecto para decidir a quién tratar.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y **ejecutables** de los ejercicios del README (sección `## 🧪 Ejercicios`). Datos sintéticos con `np.random.default_rng(42)`, sin internet. Cada bloque imprime resultados y valida con `assert`.

### Ejercicio 1 — DiD ingenuo
Panel 2×2. `β₃` (interacción `treated×post`) recupera el efecto verdadero. Si se violan las *parallel trends*, el DiD captura la divergencia como efecto espurio.

In [ ]:
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
rng = np.random.default_rng(42)
true_effect, n_unit = 3.0, 500
rows = []
for g in (0, 1):
    for post in (0, 1):
        y = 10 + 2*g + 1.5*post + true_effect*(g*post) + rng.normal(0, 2, n_unit)
        for v in y: rows.append((g, post, v))
d = pd.DataFrame(rows, columns=["treated", "post", "Y"])
did = smf.ols("Y ~ treated * post", data=d).fit().params["treated:post"]
print(f"DiD (treated:post) = {did:.3f}  (verdadero {true_effect})")
assert abs(did - true_effect) < 0.5

# Parallel trends VIOLADO: el tratado ya divergia en post SIN tratamiento real (efecto=0)
rows2 = []
for g in (0, 1):
    for post in (0, 1):
        y = 10 + 2*g + 1.5*post + 0.0*(g*post) + 3.0*(g*post) + rng.normal(0, 2, n_unit)
        for v in y: rows2.append((g, post, v))
d2 = pd.DataFrame(rows2, columns=["treated", "post", "Y"])
did2 = smf.ols("Y ~ treated * post", data=d2).fit().params["treated:post"]
print(f"DiD con parallel trends VIOLADO (efecto real=0) = {did2:.3f}  <- sesgo espurio")
assert abs(did2 - 3.0) < 0.5

### Ejercicio 2 — Event study
Panel de 10 períodos (5 pre, 5 post). Los coeficientes pre ≈ 0 avalan *parallel trends*; los post recuperan el efecto.

In [ ]:
rng = np.random.default_rng(42)
periods = np.arange(-5, 6)
n_per, effect_post = 300, 2.0
P, Gg, Y = [], [], []
for t in periods:
    for g in (0, 1):
        eff = effect_post if (g == 1 and t >= 0) else 0.0
        y = 5 + 0.3*t + 1.0*g + eff + rng.normal(0, 1, n_per)   # tendencia comun (parallel)
        P += [t]*n_per; Gg += [g]*n_per; Y += list(y)
P, Gg, Y = np.array(P), np.array(Gg), np.array(Y)
def cell(t, g): return Y[(P == t) & (Gg == g)].mean()
base = cell(-1, 1) - cell(-1, 0)                       # baseline t=-1
coefs = {t: (cell(t, 1) - cell(t, 0)) - base for t in periods if t != -1}
pre = [coefs[t] for t in periods if t < -1]
post = [coefs[t] for t in periods if t >= 0]
for t in periods:
    if t != -1: print(f"  t={t:+d}: {coefs[t]:+.3f}")
print(f"media pre  (deberia ~0) = {np.mean(pre):+.3f}")
print(f"media post (deberia ~2) = {np.mean(post):+.3f}")
assert abs(np.mean(pre)) < 0.4 and abs(np.mean(post) - 2.0) < 0.5

### Ejercicio 3 — T-learner
Dataset de email sintético (sin internet). Dos `RandomForestClassifier`, uplift = `p₁(x) - p₀(x)`. Los persuadables (X1>0) muestran mayor uplift.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rng = np.random.default_rng(42)
N = 8000
X = rng.normal(0, 1, (N, 5))
T = rng.binomial(1, 0.5, N)                       # email aleatorio
tau = 0.15*(X[:, 0] > 0)                           # persuadables: X1>0
base = 1/(1 + np.exp(-(0.5*X[:, 0] - 0.3*X[:, 1])))
prob = np.clip(base + tau*T, 0, 1)
Y = rng.binomial(1, prob)
m1 = RandomForestClassifier(150, random_state=42).fit(X[T==1], Y[T==1])
m0 = RandomForestClassifier(150, random_state=42).fit(X[T==0], Y[T==0])
uplift = m1.predict_proba(X)[:, 1] - m0.predict_proba(X)[:, 1]
print(f"uplift medio          = {uplift.mean():.3f}")
print(f"uplift medio (X1>0)   = {uplift[X[:,0]>0].mean():.3f}  <- persuadables")
print(f"uplift medio (X1<=0)  = {uplift[X[:,0]<=0].mean():.3f}")
assert uplift[X[:,0]>0].mean() > uplift[X[:,0]<=0].mean()

### Ejercicio 4 — Qini curve
`scikit-uplift` no está instalada → Qini a mano. El modelo ordena mejor que tratar al azar (área Qini positiva).

In [ ]:
def qini_curve(y, t, score):
    order = np.argsort(-score)
    y, t = y[order], t[order]
    n_t, n_c = np.cumsum(t), np.cumsum(1 - t)
    r_t, r_c = np.cumsum(y*t), np.cumsum(y*(1 - t))
    n_c_safe = np.where(n_c == 0, 1, n_c)
    return r_t - r_c*(n_t / n_c_safe)

q_model = qini_curve(Y, T, uplift)
q_rand = qini_curve(Y, T, rng.normal(0, 1, N))    # ranking aleatorio
frac = np.arange(1, N + 1) / N
qini_coef = np.trapezoid(q_model, frac) - np.trapezoid(q_rand, frac)
print(f"ganancia incremental top (modelo) = {q_model[-1]:.1f}")
print(f"area Qini (modelo - aleatorio)     = {qini_coef:.1f}")
assert qini_coef > 0

### Ejercicio 5 — Synthetic Control
`pysyncon` no está instalada → pesos convexos por optimización (`scipy`). Panel 10 estados × 20 años, tratamiento en California (año 11). Placebo in-space ubica a California en el extremo.

In [ ]:
from scipy.optimize import minimize
rng = np.random.default_rng(42)
n_states, n_years, treat_year = 10, 20, 11
time = np.arange(n_years)
common = np.sin(time / 3)
Y10 = np.zeros((n_states, n_years))
for s in range(n_states):
    Y10[s] = rng.normal(10, 2) + rng.uniform(0.5, 1.5)*common + rng.normal(0, 0.5, n_years)
# California = estado 0, efecto negativo creciente post-tratamiento
Y10[0, treat_year:] += -3.0 * (np.arange(n_years - treat_year) + 1) / (n_years - treat_year)
pre = slice(0, treat_year)
bnds = [(0, 1)]*(n_states - 1)
cons = ({"type": "eq", "fun": lambda w: w.sum() - 1},)

def fit_w(treated, donors):
    loss = lambda w: np.sum((treated[pre] - w @ donors[:, pre])**2)
    r = minimize(loss, np.ones(len(donors))/len(donors), bounds=bnds, constraints=cons, method="SLSQP")
    return r.x

donors0 = Y10[1:]
w = fit_w(Y10[0], donors0)
gap = Y10[0] - w @ donors0
print(f"pesos (pocos > 0): {np.round(w, 2)}")
print(f"pre-RMSPE = {np.sqrt(np.mean(gap[pre]**2)):.3f}  |  gap post medio = {gap[treat_year:].mean():.3f}")

def placebo(idx):
    tr = Y10[idx]; do = np.delete(Y10, idx, axis=0)
    return tr - fit_w(tr, do) @ do
post_rmspe = np.array([np.sqrt((placebo(i)[treat_year:]**2).mean()) for i in range(n_states)])
rank = int((post_rmspe >= post_rmspe[0]).sum())
print(f"post-RMSPE California rank = {rank}/{n_states} (1 = mas extremo)")
assert gap[treat_year:].mean() < -1 and rank <= 3